# 🌿 Plant Disease Detection Backend (Google Colab)
This notebook automatically downloads the **Kaggle Plant Disease Dataset (PlantVillage)** via `kagglehub`, trains a MobileNetV2 Deep Learning Transfer Learning model, and hosts a **Flask REST API** with `pyngrok` tunnel for frontend integration.

### Step 1: Install Required Libraries

In [ ]:
!pip install -q kagglehub opendatasets pyngrok flask-cors tensorflow pillow

### Step 2: Download Kaggle Dataset Automatically (No setup needed!)

In [ ]:
import os
import kagglehub

# Download latest Kaggle Plant Disease dataset automatically
dataset_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print("✅ Dataset downloaded to:", dataset_path)

# Find train & valid folders recursively
train_dir, valid_dir = None, None
for root, dirs, files in os.walk(dataset_path):
    if "train" in dirs and "valid" in dirs:
        train_dir = os.path.join(root, "train")
        valid_dir = os.path.join(root, "valid")
        break

print("Train folder:", train_dir)
print("Valid folder:", valid_dir)

### Step 3: Train Deep Learning Model (MobileNetV2 Transfer Learning)

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)
valid_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(train_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')
valid_gen = valid_datagen.flow_from_directory(valid_dir, target_size=img_size, batch_size=batch_size, class_mode='categorical')

# Save class indices mapping for API inference
class_indices = train_gen.class_indices
class_names = {v: k for k, v in class_indices.items()}
with open("class_indices.json", "w") as f:
    json.dump(class_names, f, indent=4)
print(f"Saved class_indices.json with {len(class_names)} classes!")

# Model Creation
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
preds = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=preds)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
checkpoint = ModelCheckpoint("plant_disease_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_gen,
    epochs=5,
    validation_data=valid_gen,
    callbacks=[checkpoint, early_stop]
)

print("Training Complete! Saved model to plant_disease_model.h5")

### Step 4: Run Flask Backend & Expose Public Tunnel URL

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from PIL import Image
import numpy as np
import json
import tensorflow as tf
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

# Load trained model and class labels
model = tf.keras.models.load_model("plant_disease_model.h5")
with open("class_indices.json", "r") as f:
    class_names = {int(k): v for k, v in json.load(f).items()}

def format_label(label):
    clean_name = label.replace("___", " - ").replace("_", " ")
    is_healthy = "healthy" in label.lower()
    status = "Healthy" if is_healthy else "Diseased"
    return clean_name, status

@app.route("/", methods=["GET"])
def home():
    return jsonify({"status": "online", "message": "Plant Disease Colab API is running!"})

@app.route("/predict", methods=["POST", "OPTIONS"])
def predict():
    if request.method == "OPTIONS":
        return jsonify({"status": "ok"}), 200
    
    if "image" not in request.files:
        return jsonify({"error": "No image uploaded. Form key must be 'image'."}), 400
    
    file = request.files["image"]
    if file.filename == "":
        return jsonify({"error": "No file selected."}), 400
    
    try:
        img = Image.open(file.stream).convert("RGB").resize((224, 224))
        img_arr = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)
        
        preds = model.predict(img_arr)[0]
        top_idx = int(np.argmax(preds))
        confidence = float(preds[top_idx]) * 100
        
        raw_label = class_names.get(top_idx, f"Class #{top_idx}")
        disease_name, status = format_label(raw_label)
        
        return jsonify({
            "disease": disease_name,
            "confidence": f"{confidence:.2f}%",
            "status": status
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Open Ngrok Tunnel
public_url = ngrok.connect(5000)
print("\n=======================================================")
print(f"🚀 PUBLIC BACKEND URL: {public_url.public_url}")
print(f"👉 COPY THIS PREDICT URL TO FRONTEND: {public_url.public_url}/predict")
print("=======================================================\n")

app.run(port=5000)